In [2]:
import numpy as np
import pandas as pd
import joblib
import os
import h5py

In [3]:
data_dir = "combined_scimilarity_5neuro"
output_dir = "combined_scimilarity_5neuro_shuffled"
embedding = "scimilarity"
indices_file = "shuffled_indices.npy"

metadata_file = "obs_annotated.tsv.gz"
if embedding == "uce":
    embedding_file = "uce.h5"
elif embedding == "scimilarity":
    embedding_file = "scimilarity.h5"

os.makedirs(output_dir, exist_ok=True)

In [4]:
meta_data = pd.read_csv(os.path.join(data_dir, metadata_file), sep ='\t')

/tmp/ipykernel_22707/570911309.py:1: DtypeWarning: Columns (2,3,4,5,6,7,8,9,13,14,16,17,18,19,20,21,72,73,74,75,76,78,79,80,81,82,84,85,86,88,90,91,92,94,96,97,98,100,102,103,104,108,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,131,133,134,135,136,137,138,139,140,141,146,147,151) have mixed types. Specify dtype option on import or set low_memory=False.
  meta_data = pd.read_csv(os.path.join(data_dir, metadata_file), sep ='\t')


## LOAD indices

In [ ]:
row_indices = np.load(os.path.join(data_dir, indices_file))
row_indices.shape

## Extract new metadata using those indices

In [25]:
new_meta_data = meta_data.iloc[row_indices]

In [26]:
new_meta_data.to_csv(os.path.join(output_dir, metadata_file), sep="\t", index=False)

## Extract embedding using those indices

In [15]:
# handle to uce.h5 
fembedding = h5py.File(os.path.join(data_dir, embedding_file), "r")
embedding = fembedding["data"] # h5 handle
print(embedding.shape)

(7100746, 128)


In [16]:
def smart_h5py_batch_read(dataset, sorted_indices):
    """Efficiently read sorted indices from HDF5 by grouping nearby rows."""
    output = []
    is_sorted = np.all(sorted_indices[:-1] <= sorted_indices[1:])
    assert(is_sorted)
    #sorted_indices = np.sort(np.array(sorted_indices, dtype=int))  # Explicit sort
    i = 0
    while i < len(sorted_indices):
        start = sorted_indices[i]
        j = i
        while ( j + 1 < len(sorted_indices) and sorted_indices[j + 1] == sorted_indices[j] + 1):
            j += 1
        end = sorted_indices[j]
        chunk = dataset[start:end+1, :] # Read slice
        output.append(chunk)
        i = j + 1
    return np.vstack(output)

In [ ]:
output_h5 = os.path.join(output_dir, embedding_file)
total_len = len(row_indices)
minibatch_size = 1000
        
with h5py.File(output_h5, "w") as f:
    start_idx = 0
    dset = f.create_dataset(
        "data", shape= embedding.shape, dtype=embedding.dtype
    )

    for i in range(0, total_len, minibatch_size):
        if i % 20_000 == 0:
            print("Processing cell", start_idx)
        idx_batch = row_indices[i:i+minibatch_size]
        sort_idx = np.argsort(idx_batch)
        idx_batch_sorted = idx_batch[sort_idx]
        #idx_batch_sorted = row_indices[i:i+minibatch_size]
        batch_sorted = smart_h5py_batch_read(embedding, idx_batch_sorted)
        # invert the permutation to restore original order
        unsort_idx = np.argsort(sort_idx)
        batch = batch_sorted[unsort_idx]

        end_idx = start_idx + batch.shape[0]
        dset[start_idx:end_idx, :] = batch
        start_idx = end_idx

In [24]:
fembedding.close()